Cleaning

In [42]:
import pandas as pd
import numpy as np
import os

sumber_input = 'Dataset/DoveR_2019Raw.csv'
sumber_output = 'Dataset/DoveR_2019.csv'

target_input = 'Dataset/SuperDove_2024Raw.csv'
target_output = 'Dataset/SuperDove_2024.csv'


#domain sumber
if os.path.exists(sumber_input):
    df1 = pd.read_csv(sumber_input)
    df1.replace('=-nan(ind)', np.nan, inplace=True)
    for col in df1.select_dtypes(include=['object']).columns:
        if 'VALUE' not in col:
            df1[col] = pd.to_numeric(df1[col], errors='coerce')
    df1.dropna(inplace=True)
    if 'VALUE' in df1.columns:
        df1['VALUE'] = df1['VALUE'] - 1
    df1.to_csv(sumber_output, index=False)


#domain target
if os.path.exists(target_input):
    df2 = pd.read_csv(target_input)
    df2.replace('=-nan(ind)', np.nan, inplace=True)
    for col in df2.select_dtypes(include=['object']).columns:
        if 'VALUE' not in col:
            df2[col] = pd.to_numeric(df2[col], errors='coerce')
    df2.dropna(inplace=True)
    if 'VALUE' in df2.columns:
        df2['VALUE'] = df2['VALUE'] - 1
    df2.to_csv(target_output, index=False)

Distribusi Dataset

In [38]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATASETS = {
    "Domain Sumber (Dove-R 2019)": ("Dataset/DoveR_2019.csv", "distribusi_sumber.png"),
    "Domain Target (SuperDove 2024)": ("Dataset/Superdove_2024.csv", "distribusi_target.png")
}
OUTPUT_DIR = "hasil_analisis_data"

CLASS_MAPPING = {
    0: "Bangunan", 1: "Jalan", 2: "Ladang",
    3: "Lahan Terbuka", 4: "Sawah", 5: "Perkebunan"
}
CLASS_COLORS = {
    "Bangunan": "#b3e5fc", "Jalan": "#e1bee7", "Ladang": "#fff59d",
    "Lahan Terbuka": "#ffccbc", "Sawah": "#c8e6c9", "Perkebunan": "#9fa8da"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

for title, (csv_path, out_file) in DATASETS.items():
    df = pd.read_csv(csv_path)
    if "VALUE" not in df.columns:
        continue

    counts = df["VALUE"].value_counts().sort_index()
    valid_idx = [i for i in counts.index if i in CLASS_MAPPING]
    names = [CLASS_MAPPING[i] for i in valid_idx]
    values = [counts[i] for i in valid_idx]
    colors = [CLASS_COLORS[n] for n in names]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, values, color=colors)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                int(bar.get_height()), ha="center", va="bottom", fontsize=9)

    ax.set_title(title, fontsize=13)
    ax.set_xticklabels(names, rotation=25, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, out_file), dpi=300)
    plt.close(fig)


C:\Users\Kietzz\AppData\Local\Temp\ipykernel_21752\1871744001.py:40: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=25, ha="right")
C:\Users\Kietzz\AppData\Local\Temp\ipykernel_21752\1871744001.py:40: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=25, ha="right")


Split Data Latih & Uji

In [39]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SOURCE_CSV_PATH = "Dataset/DoveR_2019.csv"
TARGET_CSV_PATH = "Dataset/Superdove_2024.csv"
OUT_SPLIT_DIR   = "Dataset/Splitted"
TEST_SPLIT_SIZE = 0.30
FT_TRAIN_SIZE   = 0.20
RANDOM_STATE    = 42

os.makedirs(OUT_SPLIT_DIR, exist_ok=True)

df_src = pd.read_csv(SOURCE_CSV_PATH)
df_tgt = pd.read_csv(TARGET_CSV_PATH)

df_src_tr, df_src_te = train_test_split(
    df_src, test_size=TEST_SPLIT_SIZE,
    stratify=df_src["VALUE"], random_state=RANDOM_STATE
)
df_tgt_tr, df_tgt_te = train_test_split(
    df_tgt, train_size=FT_TRAIN_SIZE,
    stratify=df_tgt["VALUE"], random_state=RANDOM_STATE
)

cols_src = ["VALUE"] + [c for c in df_src.columns if c != "VALUE"]
cols_tgt = ["VALUE"] + [c for c in df_tgt.columns if c != "VALUE"]

df_src_tr[cols_src].to_csv(os.path.join(OUT_SPLIT_DIR, "source_train.csv"), index=False)
df_src_te[cols_src].to_csv(os.path.join(OUT_SPLIT_DIR, "source_test.csv"), index=False)
df_tgt_tr[cols_tgt].to_csv(os.path.join(OUT_SPLIT_DIR, "target_train.csv"), index=False)
df_tgt_te[cols_tgt].to_csv(os.path.join(OUT_SPLIT_DIR, "target_test.csv"), index=False)

{
    "source_train": df_src_tr.shape,
    "source_test":  df_src_te.shape,
    "target_train": df_tgt_tr.shape,
    "target_test":  df_tgt_te.shape,
}


{'source_train': (22451, 14),
 'source_test': (9622, 14),
 'target_train': (4114, 14),
 'target_test': (16458, 14)}

Plot Latih vs Uji

In [40]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

SOURCE_TRAIN_CSV_PATH = "Dataset/Splitted/source_train.csv"
SOURCE_TEST_CSV_PATH  = "Dataset/Splitted/source_test.csv"
TARGET_TRAIN_CSV_PATH = "Dataset/Splitted/target_train.csv"
TARGET_TEST_CSV_PATH  = "Dataset/Splitted/target_test.csv"
OUTPUT_DIR = "hasil_analisis_data"

CLASS_MAPPING = {
    0: "Bangunan", 1: "Jalan", 2: "Ladang",
    3: "Lahan Terbuka", 4: "Sawah", 5: "Perkebunan"
}
CLASS_COLORS = {
    "Bangunan": "#b3e5fc", "Jalan": "#e1bee7", "Ladang": "#fff59d",
    "Lahan Terbuka": "#ffccbc", "Sawah": "#c8e6c9", "Perkebunan": "#9fa8da"
}

def _count_by_class(df, classes_order):
    counts = df["VALUE"].value_counts().sort_index()
    return counts.reindex(classes_order, fill_value=0).astype(int)

def visualize_distribution_pair(df_train, df_test, title, filename):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    classes_order = list(CLASS_MAPPING.keys())
    class_labels = [CLASS_MAPPING[c] for c in classes_order]
    train_counts = _count_by_class(df_train, classes_order)
    test_counts = _count_by_class(df_test, classes_order)
    x = np.arange(len(classes_order))
    width = 0.38
    base_colors = [CLASS_COLORS[name] for name in class_labels]
    fig, ax = plt.subplots(figsize=(12, 7))
    bars_train = ax.bar(x - width/2, train_counts.values, width,
                        color=base_colors, alpha=0.95, edgecolor="black", linewidth=0.7,
                        label="Train")
    bars_test = ax.bar(x + width/2, test_counts.values, width,
                       color=base_colors, alpha=0.55, edgecolor="black", linewidth=0.7,
                       label="Test")
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Kelas Penutup dan Penggunaan Lahan", fontsize=12)
    ax.set_ylabel("Jumlah Sampel Piksel", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=0)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    for bars in [bars_train, bars_test]:
        for bar in bars:
            yval = bar.get_height()
            if yval > 0:
                ax.text(bar.get_x() + bar.get_width()/2, yval,
                        f"{int(yval)}", va="bottom", ha="center", fontsize=9)
    ax.legend(loc="upper right", frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close(fig)

source_train_df = pd.read_csv(SOURCE_TRAIN_CSV_PATH)
source_test_df = pd.read_csv(SOURCE_TEST_CSV_PATH)
visualize_distribution_pair(source_train_df, source_test_df,
    "Distribusi Kelas Domain Sumber (Train vs Test)", "distribusi_sumber_train_test.png")

target_train_df = pd.read_csv(TARGET_TRAIN_CSV_PATH)
target_test_df = pd.read_csv(TARGET_TEST_CSV_PATH)
visualize_distribution_pair(target_train_df, target_test_df,
    "Distribusi Kelas Domain Target (Train vs Test)", "distribusi_target_train_test.png")


Separabilitas (LDA)

In [41]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

CLASS_COL = "VALUE"
OUTPUT_DIR = "hasil_analisis_data"

class_mapping = {
    0: "Bangunan", 1: "Jalan", 2: "Ladang",
    3: "Lahan Terbuka", 4: "Sawah", 5: "Perkebunan"
}
custom_palette = {
    "Sawah": "#1f77b4", "Ladang": "#ff7f0e", "Perkebunan": "#2ca02c",
    "Lahan Terbuka": "#d62728", "Jalan": "#9467bd", "Bangunan": "#8c564b"
}
legend_order = list(custom_palette.keys())

def run_lda(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df[CLASS_COL].isin(class_mapping.keys())].copy()
    X = df.drop(columns=[CLASS_COL])
    y_txt = df[CLASS_COL].map(class_mapping)
    X_scaled = StandardScaler().fit_transform(X)
    X_lda = LinearDiscriminantAnalysis(n_components=2).fit_transform(X_scaled, y_txt)
    lda_df = pd.DataFrame(X_lda, columns=["LD1", "LD2"])
    lda_df["Kelas"] = pd.Categorical(y_txt, categories=legend_order, ordered=True)
    return lda_df

lda_src = run_lda("Dataset/Splitted/source_train.csv")
lda_tgt = run_lda("Dataset/Splitted/target_train.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.scatterplot(data=lda_src, x="LD1", y="LD2", hue="Kelas",
                palette=custom_palette, hue_order=legend_order,
                s=50, alpha=0.7, ax=axes[0])
axes[0].set_title("LDA — Domain Sumber (2019)", fontsize=14)
axes[0].grid(True, linestyle="--", alpha=0.6)

sns.scatterplot(data=lda_tgt, x="LD1", y="LD2", hue="Kelas",
                palette=custom_palette, hue_order=legend_order,
                s=50, alpha=0.7, ax=axes[1])
axes[1].set_title("LDA — Domain Target (2024)", fontsize=14)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "lda_sumber_target.png"), dpi=300)
plt.close()
